In [2]:
from pyspark.sql import functions as F

# ============================================================
# NB_Gold_Fact_Orders
# Silver -> Gold
# Grain: one row per order
# ============================================================

ORDERS_TABLE = "silver_orders"
CUSTOMER_DIM = "dim_customer"
DATE_DIM = "dim_date"
TARGET_TABLE = "fact_orders"

# ------------------------------------------------------------
# 1. Read source and dimensions
# ------------------------------------------------------------

orders = spark.table(ORDERS_TABLE)

customers = (
    spark.table(CUSTOMER_DIM)
    .select(
        "customer_key",
        "customer_id"
    )
)

dates = (
    spark.table(DATE_DIM)
    .select(
        "date_key",
        "date"
    )
)

# ------------------------------------------------------------
# 2. Derive order date
# ------------------------------------------------------------

orders = orders.withColumn(
    "order_date_only",
    F.to_date("order_date")
)

# ------------------------------------------------------------
# 3. Join customer dimension
# ------------------------------------------------------------

orders = (
    orders
    .join(
        customers,
        on="customer_id",
        how="left"
    )
)

# ------------------------------------------------------------
# 4. Join date dimension
# ------------------------------------------------------------

orders = (
    orders
    .join(
        dates,
        orders.order_date_only == dates.date,
        how="left"
    )
)

# ------------------------------------------------------------
# 5. Build Gold fact
# ------------------------------------------------------------

df_gold = (
    orders
    .select(
        "order_id",
        "customer_key",
        "date_key",
        "order_date",
        "status",
        "total_amount"
    )
    .withColumn(
        "order_count",
        F.lit(1)
    )
)

# ------------------------------------------------------------
# 6. Write Gold fact
# ------------------------------------------------------------

df_gold.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(TARGET_TABLE)

# ------------------------------------------------------------
# 7. Validation
# ------------------------------------------------------------

print(f"Gold table created: {TARGET_TABLE}")
print(f"Records: {df_gold.count()}")

df_gold.show(10, truncate=False)

StatementMeta(, 5acafe2d-51da-4093-ad73-e08abf93f073, 4, Finished, Available, Finished, False)

Gold table created: fact_orders
Records: 49980
+--------+------------+--------+-------------------+---------+------------+-----------+
|order_id|customer_key|date_key|order_date         |status   |total_amount|order_count|
+--------+------------+--------+-------------------+---------+------------+-----------+
|1       |1904        |20250327|2025-03-27 02:28:32|delivered|2426.08     |1          |
|2       |1400        |20260218|2026-02-18 12:45:15|pending  |2954.74     |1          |
|4       |7461        |20250424|2025-04-24 03:25:26|cancelled|2415.11     |1          |
|5       |7192        |20260914|2026-09-14 09:32:32|delivered|225.42      |1          |
|6       |1786        |20250322|2025-03-22 14:52:59|delivered|3274.61     |1          |
|7       |4835        |20251031|2025-10-31 04:20:57|delivered|55.74       |1          |
|8       |9265        |20260621|2026-06-21 09:38:12|cancelled|4980.86     |1          |
|9       |9488        |20241115|2024-11-15 12:04:38|shipped  |2435.36    